# 01 · The four uncertainty signals (≈ 12 min)
**Challenge 2: build a self-consistency detector.**

For each question we stored the model's *greedy answer* (the one a user would see) and **10 sampled answers**.
Four **uncertainty signals** (higher = more uncertain) are computed from that evidence:

| Signal | Abbreviation | Measures |
|---|---|---|
| Log-probability | LP | the model's own token confidence in the greedy answer |
| Self-consistency | SC | do the 10 samples give the same *string*? |
| Semantic entropy | SE | do the 10 samples give the same *meaning*? |
| NLI contradiction | NLI | do the samples contradict the greedy answer? (NLI = natural language inference) |

The heavy model calls were done in advance, so everything here runs on your laptop.

In [ ]:
import sys, pathlib, warnings
from collections import Counter
REPO = "pycon2026-uncertainty-workshop"   # <- must match your GitHub repo name (used on Colab only)
if "google.colab" in sys.modules:
    ROOT = pathlib.Path("/content") / REPO
    if not ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone", f"https://github.com/Shafiya0101/{REPO}.git", str(ROOT)], check=True)
else:
    ROOT = pathlib.Path.cwd()
    if ROOT.name in ("notebooks", "solutions"):
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from uncertainty_lib import *
from uncertainty_lib import checks
from uncertainty_lib.plotting import set_style, plot_region_grid, plot_risk_coverage
set_style()
pd.set_option("display.max_colwidth", 70)

In [ ]:
df = load_setting("mistral_triviaqa")
checks.banner(df)
df["cluster_ids"] = df["bidir_entail"].apply(greedy_cluster)   # meaning clusters, built from stored NLI results
row = df.iloc[0]
print("Question :", row.question)
print("Gold     :", row.gold)
print("Greedy   :", row.greedy)
print("Samples  :")
for s in row.samples:
    print("   -", s)

## Step 1 · Self-consistency (SC)
`U_SC = 1 − (size of the largest group of identical answers) / N`.
10/10 identical → 0.0. Five different answers, two each → 0.8.

**Predict first:** `Car Seats`, `car seats`, `The Car Seats.` and `Car Seats (Automobile Interiors)`: which of these end up in the same group? (The helper `normalize()` decides; try it.)

In [ ]:
for a in ["Car Seats", "car seats", "The Car Seats.", "Car Seats (Automobile Interiors)"]:
    print(repr(a), "->", repr(normalize(a)))

### 🛠 TODO 1: implement `my_self_consistency`

In [ ]:
def my_self_consistency(samples):
    groups = Counter(normalize(s) for s in samples)
    return 1 - max(groups.values()) / len(samples)

In [ ]:
checks.check_sc(my_self_consistency, self_consistency_uncertainty, df)

## Step 2 · Semantic entropy (SE)
SC compares *strings*, so a paraphrase looks like disagreement. SE first groups samples by **meaning**
(`cluster_ids`: samples with the same id mean the same thing), then measures how spread the samples are over those groups:

`U_SE = − Σ p_k log p_k`, with `p_k` = share of samples in cluster k.

### 🛠 TODO 2: implement `my_semantic_entropy`

In [ ]:
def my_semantic_entropy(cluster_ids):
    counts = np.array(list(Counter(cluster_ids).values()), dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum())

In [ ]:
checks.check_se(my_semantic_entropy, semantic_entropy, df)

## Step 3 · All four signals
`add_signals` computes LP, SC, SE and NLI for every question, plus two averages: **Comb3** (LP+SC+SE) and **Comb4** (+NLI).

In [ ]:
df = add_signals(df)
df[["greedy", "label_containment", "U_lp", "U_sc", "U_se", "U_nli"]].round(2).head(6)

### Where SC and SE disagree
A question where the *strings* differ (SC high) but the *meaning* is stable (SE low): a paraphrase, not real doubt.

In [ ]:
cand = df[(df.U_sc >= 0.5) & (df.U_se <= 0.35)]
if len(cand):
    r = cand.iloc[0]
    print(f"U_SC = {r.U_sc:.2f}   U_SE = {r.U_se:.2f}")
    for s in r.samples:
        print("   -", s)
else:
    print("No such question in this setting. Try another one.")

## Step 4 · Which signal detects wrong answers best?
**Predict before you run:** which of the six scores has the highest AUROC?
(AUROC = chance that a random *wrong* answer gets a higher uncertainty than a random *correct* one. 0.5 = coin flip, 1.0 = perfect.)

In [ ]:
auc = auroc_table(df)
for k, v in auc.items():
    print(f"{k:7s} AUROC = {v:.3f}")
fig, ax = plt.subplots()
ax.barh(list(auc), list(auc.values()), color="0.55")
for i, v in enumerate(auc.values()):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center")
ax.axvline(0.5, ls="--", color="black"); ax.text(0.5, -0.9, "chance", ha="center")
ax.set_xlim(0.4, 1); ax.set_xlabel("AUROC (higher = better)"); ax.set_title("Detecting wrong answers")
plt.show()

**Discuss with your neighbour (1 min):** are the differences between signals large? With 200 questions, a gap of 0.02 is mostly noise. Keep that in mind for notebook 03.